# Task Comparison: PSDs and Topomaps

## Overview

You will convert task markers into 15-second epochs, compare condition spectra, and map alpha power. The cells provide structure and API hints, but you must implement the analysis and check each intermediate result.

## Learning objectives

- Standardize task-marker variants without changing their timing.
- Convert annotations into events and exact 15-second epochs.
- Compute and compare Welch PSDs.
- Summarize 8–12 Hz alpha activity for each EEG channel.
- Make comparable condition and difference topomaps.

## Scientific background: eyes-closed alpha

Alpha is an approximately 8–12 Hz rhythm often strongest over posterior scalp during quiet wakefulness. Closing the eyes commonly increases posterior alpha power. This is an expected tendency, not a guaranteed individual result. Drowsiness, muscle artifacts, reference choice, marker errors, and electrode layout can alter the pattern.

In [ ]:
# Setup cell: run this as provided.
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import mne
import numpy as np
import pandas as pd
import pyxdf

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))
from src.eeg_utils import extract_marker_annotations, summarize_xdf_streams

mne.set_log_level("WARNING")

## Step 1: Load preprocessed EEG

Load the FIF file with preload enabled. Print duration, sampling rate, EEG channel count, bad channels, and annotation count. Confirm that these values agree with earlier notebooks before analyzing conditions.

In [ ]:
PREPROCESSED_PATH = Path("../outputs/raw_preprocessed.fif")

# TODO: load the preprocessed Raw object into memory.
raw = ...

# TODO: print the requested quality-control information.
...

## Step 2: Load or recreate task annotations

First inspect `raw.annotations.description` with `value_counts`. If task markers were saved in Notebook 1, use them. If not, reload the original XDF, display its stream summary, select EEG and marker indices, and call `extract_marker_annotations(marker_stream, raw_start_time=first_eeg_timestamp)`.

Marker spelling often varies. The accepted variants below are deliberately editable. Normalize strings only for matching; do not change timestamps. Build new annotations whose descriptions are exactly `eyes_open` or `eyes_closed`, with 15-second durations. Preserve annotations beginning with `BAD` or `EDGE` so MNE can reject affected epochs.

In [ ]:
EYES_OPEN_LABELS = ["eyes_open", "open", "EO"]
EYES_CLOSED_LABELS = ["eyes_closed", "closed", "EC"]
XDF_PATH = "../data/sample_recording.xdf"

# TODO: inspect current annotation labels and counts.
...

# Optional fallback if task annotations are missing:
# streams, _ = pyxdf.load_xdf(XDF_PATH)
# display(summarize_xdf_streams(streams))
# EEG_STREAM_INDEX = ...
# MARKER_STREAM_INDEX = ...
# eeg_start = streams[EEG_STREAM_INDEX]['time_stamps'][0]
# raw.set_annotations(extract_marker_annotations(..., raw_start_time=...))

In [ ]:
def normalize_marker(text):
    """Return a marker string suitable for case-insensitive matching."""
    # TODO: convert to string, remove surrounding whitespace, and ignore case.
    return ...

# TODO: convert both label lists to sets of normalized strings.
open_variants = ...
closed_variants = ...

canonical_onsets = []
canonical_descriptions = []
for annotation in raw.annotations:
    # TODO: normalize the description and append the onset/canonical label
    # only when it matches one of the accepted condition variants.
    ...

# TODO: build 15-second task annotations from the collected lists.
task_annotations = mne.Annotations(
    onset=...,
    duration=...,
    description=...,
)

# TODO (challenge): preserve existing BAD/EDGE annotations.
# quality_annotations = ...

## Step 3: Convert annotations to events

Set the canonical annotations on a copy of `raw`. Define explicit codes `eyes_open: 1` and `eyes_closed: 2`, then use `mne.events_from_annotations`. Count both event codes. The expected design has 20 per condition; do not silently continue if counts differ. Investigate missing, extra, or misspelled markers.

In [ ]:
# TODO: attach canonical task annotations (and quality annotations if present).
raw.set_annotations(...)

event_id = {"eyes_open": 1, "eyes_closed": 2}

# TODO: convert annotations to events using the explicit event_id.
events, returned_event_id = ...

# TODO: count events for each condition and compare with 20 each.
event_counts = {...}
print(event_counts)

## Step 4: Create eyes-open and eyes-closed epochs

Create epochs beginning at each marker. To obtain exactly 15 seconds of samples, use `tmin=0` and `tmax=15 - 1/sfreq`. Set `baseline=None`, select EEG channels, preload, and reject segments overlapped by BAD annotations. Print retained counts and inspect `epochs.drop_log`.

In [ ]:
TRIAL_DURATION = 15.0

# TODO: calculate an inclusive MNE tmax for exactly 15 seconds.
tmax = ...

# TODO: create the epochs with the parameters described above.
epochs = mne.Epochs(
    raw=...,
    events=...,
    event_id=...,
    tmin=...,
    tmax=...,
    baseline=...,
    picks=...,
    preload=...,
    reject_by_annotation=...,
)

# TODO: print retained counts for both conditions.
...

## Step 5: Plot example epochs

Plot three eyes-open and three eyes-closed epochs using MNE condition selection (`epochs['condition']`). Inspect whether one large artifact could dominate a condition average. Record any trial exclusions using a rule that does not depend on condition.

In [ ]:
# TODO: select and plot up to three epochs from each condition.
epochs[...][:3].plot(...)
epochs[...][:3].plot(...)

## Step 6: Compare PSDs

MNE can calculate and plot the power spectral density (PSD) for us. Use `epochs[condition].compute_psd(method='welch', fmin=1, fmax=40)` for each condition, then call the spectrum's `.plot(average=True)` method. MNE handles the Welch settings, averaging, units, and display scaling.

Make two plots side by side with a shared y-axis. Look for a larger bump near 8–12 Hz in the eyes-closed plot. Keeping the same frequency range and shared y-axis makes the comparison fair.

In [ ]:
# TODO: compute a 1–40 Hz spectrum for each condition.
psd_open = epochs[...].compute_psd(method="welch", fmin=..., fmax=...)
psd_closed = epochs[...].compute_psd(method="welch", fmin=..., fmax=...)

# TODO: let MNE plot the average EEG spectrum for each condition.
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
psd_open.plot(average=True, picks="eeg", axes=axes[0], show=False)
psd_closed.plot(average=True, picks="eeg", axes=axes[1], show=False)
axes[0].set_title("Eyes open")
axes[1].set_title("Eyes closed")
plt.tight_layout()

## Step 7: Compute alpha-band power

A PSD contains many frequency values for every channel. A topomap needs just **one number per channel**, so we need a simple alpha summary.

Compute a second spectrum restricted to 8–12 Hz. `get_data()` has the shape trials × channels × frequencies. Taking the mean across trials (axis 0) and frequencies (axis 2) leaves one mean alpha PSD value per channel. Multiplying by `1e12` converts V²/Hz to the more readable µV²/Hz.

This tutorial uses the mean PSD within the alpha range. Integrating the curve would estimate total alpha-band power, but that additional numerical concept is not needed to compare conditions here.

In [ ]:
# TODO: compute spectra containing only the 8–12 Hz alpha range.
alpha_psd_open = epochs[...].compute_psd(method="welch", fmin=..., fmax=...)
alpha_psd_closed = epochs[...].compute_psd(method="welch", fmin=..., fmax=...)

# TODO: average over trials and frequency bins, leaving one value per channel.
# The factor 1e12 converts V²/Hz to µV²/Hz.
alpha_open_mean = alpha_psd_open.get_data().mean(axis=(..., ...)) * 1e12
alpha_closed_mean = alpha_psd_closed.get_data().mean(axis=(..., ...)) * 1e12
alpha_difference = ...

# TODO: display a channel table to see where the largest differences occur.
alpha_table = pd.DataFrame(
    {"eyes_open": ..., "eyes_closed": ..., "closed_minus_open": ...},
    index=epochs.ch_names,
)
display(alpha_table.sort_values("closed_minus_open", ascending=False))

## Step 8: Plot alpha topomaps

Topomaps need electrode locations. Instead of inspecting coordinate arrays, first call `epochs.plot_sensors(show_names=True)`. You should see the channel names arranged sensibly around a head. If MNE reports missing locations—or channels appear piled together—return to Notebook 2 and set the correct montage.

Once the sensor plot looks correct, use the spectrum object's built-in `.plot_topomap()` method. Pass an alpha band dictionary such as `{'Alpha (8–12 Hz)': (8, 12)}`. MNE will average the alpha frequencies and draw the scalp map. Make one map for each condition and read each colorbar before comparing colors.

In [ ]:
# Visual check: do the labels appear in sensible locations on the head?
epochs.plot_sensors(show_names=True)

alpha_band = {"Alpha (8–12 Hz)": (8, 12)}

# TODO: use MNE's built-in topomap method for each alpha spectrum.
alpha_psd_open.plot_topomap(
    bands=..., ch_type="eeg", cmap="viridis"
)
alpha_psd_closed.plot_topomap(
    bands=..., ch_type="eeg", cmap="viridis"
)

## Step 9: Plot eyes-closed minus eyes-open alpha difference

The two condition maps may use different automatic color scales, so the clearest direct comparison is a difference map. Subtract eyes-open from eyes-closed mean alpha PSD for every channel.

Use `mne.viz.plot_topomap` with a red–blue color map. Set equal negative and positive limits using the largest absolute difference so zero is the center: red indicates stronger eyes-closed alpha and blue indicates stronger eyes-open alpha. A posterior red region supports the hypothesis descriptively; it is not a statistical significance test.

In [ ]:
# TODO: center the color scale on zero using the largest absolute difference.
difference_limit = np.abs(...).max()
fig, ax = plt.subplots(figsize=(5, 4))
image, _ = mne.viz.plot_topomap(
    ..., epochs.info, axes=ax, cmap="...",
    vlim=(..., ...), show=False
)
ax.set_title("Eyes closed − eyes open alpha")
fig.colorbar(image, ax=ax, label="Mean alpha PSD difference (µV²/Hz)")

# TODO: save the epochs for Notebook 4.
EPOCHS_PATH = Path("../outputs/eyes_open_closed_epo.fif")
...

## Student exercises

1. Compare a posterior-channel PSD with a frontal-channel PSD.
2. Repeat the alpha summary with 8–13 Hz and document whether the conclusion changes.
3. Plot trial-level alpha values for the strongest-difference channel and identify outliers without deleting them.

## Reflection questions

- Why might eyes-closed trials increase alpha power?
- Which channels showed the strongest difference? Were they posterior?
- Did a few trials dominate the average?
- What timing or label error could create a misleading condition difference?

## Summary

After completing the TODOs, you will have built condition epochs from real markers, checked trial counts, used MNE's built-in PSD plots, summarized alpha activity, and created condition and difference scalp maps.